# Introduction to Retrieval-Augmented Generation (RAG)

RAG combines retrieval systems with generative models to improve how NLP tasks are handled. Decoder-only models like ChatGPT generate text purely from patterns learned during training, which makes them prone to factual errors and gaps on domain-specific or rare knowledge — they have no way to look anything up. RAG fixes this by adding a retrieval step: before generating a response, the model dynamically pulls relevant information from an external knowledge source and uses it as context, rather than relying solely on what it memorized during training.

The architecture (fig below) breaks this into distinct stages: **Query Processing** prepares the user's question, the **Knowledge Base** and its **Document Embeddings** are searched by the **Retrieval System**, the **Document Retriever** pulls back the matching material as **Context**, that context is folded into a **Prompt**, and the **Decoder Model** uses it to produce the final **Generated Response**.

## What we're going to do

This notebook builds that pipeline end to end — a baseline RAG system implementing each stage in the diagram, as the starting point before layering on the more advanced techniques (hybrid/keyword search, query transformations, corrective retrieval) explored in the other notebooks in this repo.

![RAG Architecture](../images/ch06__image001.png)

## Why We Need Chunking

Embedding models turn a piece of text into one fixed-size vector by pooling across every token they read. That creates two independent problems if you try to embed a whole document as a single unit instead of splitting it up first:

1. **Context window overflow.** Every embedding model has a fixed maximum input length (e.g. `nomic-embed-text-v1.5` caps at 8,192 tokens, ~6,000 words). Feed it something longer and it doesn't error — it silently **truncates**, discarding everything past the limit before the model ever sees it. If the answer to a future query lives past that cutoff, the embedding was never computed from it, so no similarity search will ever surface that document for that query — not because retrieval failed, but because the information was thrown away before embedding happened.
2. **Semantic dilution.** Even when a document *does* fit, cramming many unrelated ideas into one vector blends them together via pooling. A chunk covering three different topics produces a vector that's an average of all three — "close" to none of them in vector space. A query about any single topic scores lower against this diluted vector than it would against a chunk that talked about only that topic. Chunking keeps each vector's "topic" narrow enough for similarity search to actually tell chunks apart.

The tradeoff: chunk too small (single sentences) and you lose the surrounding context a reader needs to make sense of the fragment. Chunk too large and you're back to dilution. There's no universal right size — it's tuned per corpus, which is what `5. Indexing` explores directly (granular vs. coarse chunking).

## Recursive Chunking

The naive way to chunk is **fixed-size splitting**: cut the text every N characters (with some overlap). It's simple, but it cuts blindly — it will happily slice a sentence in half if the boundary lands mid-word, destroying the exact local coherence chunking is supposed to preserve.

**Recursive chunking** fixes this by splitting on a *prioritized list of separators*, from "biggest structural break" down to "smallest," and only falling back to a smaller separator when the current one isn't enough to get chunks under the target size. A typical separator hierarchy:

1. `"\n\n"` — paragraph breaks (try this first; it respects the author's own structure)
2. `"\n"` — line breaks
3. `". "` — sentence boundaries
4. `" "` — word boundaries
5. `""` — raw characters (last resort, only if nothing above worked)

**The "recursive" part:** after splitting on separator 1, any resulting piece that's *still* too large gets split again — on separator 2, then 3, and so on — recursively, until every piece fits under the target chunk size. A short paragraph might need no further splitting at all; a long one might recurse all the way down to sentence-level. This is why the technique keeps as much natural structure as possible (whole paragraphs where they fit) while still guaranteeing a hard size limit everywhere else.

In practice, this is exactly what LangChain's `RecursiveCharacterTextSplitter` does — and `langchain-text-splitters` is already a dependency in this repo's `requirements.txt`:

```python
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # target max characters per chunk
    chunk_overlap=50,     # characters repeated between consecutive chunks
    separators=["\n\n", "\n", ". ", " ", ""],  # tried in this order
)

chunks = splitter.split_text(long_document_text)
```

Compare this to `5. Indexing`'s `TokenTextSplitter`, which chunks by token count rather than character count and separator hierarchy — a good next read once this section makes sense

## Loading the Source Documents

Before anything can be chunked, embedded, or retrieved, we need a corpus. This notebook uses a Kaggle dataset of research paper abstracts (`dblp-v10`), pulled via `kagglehub`.

Each row becomes a dict with two parts: `page_content` (the abstract itself — the text we'll actually chunk and embed) and `metadata` (title, authors, year, venue, and paper id — information we want to keep attached to the text for citations later, but don't want mixed into the embedding). This `page_content`/`metadata` split is exactly the shape LangChain's document utilities expect as separate `texts` and `metadatas` lists in the next step.

In [15]:
import os

import pandas as pd
import kagglehub

kagglehub.login()

path = kagglehub.dataset_download("nechbamohammed/research-papers-dataset")
df = pd.read_csv(os.path.join(path, "dblp-v10.csv"))

df = df[:500].dropna(subset=['abstract']).copy()

data = []
for row_num, row in df.iterrows():
    if row['abstract'] != 'NaN':
        data.append({
            "page_content": row['abstract'],
            "metadata": {
                "source": row["title"],
                "authors": row["authors"],
                "year": row["year"],
                "venue": row["venue"],
                "paper_id": row["id"]
            }
        })

print(f'You have {len(data)} document(s) in your data')

You have 479 document(s) in your data


## Applying Recursive Chunking to Our Documents

Now that we have raw abstracts as `page_content`/`metadata` dicts, we apply the `RecursiveCharacterTextSplitter` described above. Abstracts are short and already single-topic, so most won't need splitting at all — a larger `chunk_size` (1024 characters) is used here than in the illustrative example earlier, since we're not fighting dilution the way we would with full paper bodies.

`splitter.create_documents(...)` propagates each source document's `metadata` onto every chunk it produces, so a paper split into three chunks still has all three traceable back to the same `paper_id`, `title`, and `authors` — this is what lets us cite sources correctly once retrieval and generation are wired up.

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def recursive_split_documents(data, chunk_size=1024, chunk_overlap=50, separators=None):
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=separators,
    )

    return splitter.create_documents(
        texts=[d["page_content"] for d in data],
        metadatas=[d["metadata"] for d in data],
    )

splits = recursive_split_documents(data[:100])
print(f'You have {len(splits)} chunk(s) from {len(data)} document(s)')

You have 132 chunk(s) from 479 document(s)


## Choosing an Embedding Model

To search chunks by meaning rather than exact keywords, each one needs to become a vector. We use `nomic-embed-text-v1.5` loaded directly through Hugging Face `transformers` (tokenizer + model), rather than calling an embeddings API — this keeps the pipeline free of per-call cost and able to run fully offline once the model weights are cached locally.

`trust_remote_code=True` is required because Nomic ships custom modeling code alongside the weights rather than relying only on a stock `transformers` architecture.

In [17]:
from transformers import AutoTokenizer, AutoModel
import torch
 
text_tokenizer = AutoTokenizer.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)
text_model = AutoModel.from_pretrained("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

<All keys matched successfully>


## Generating Embeddings for Each Chunk

`get_text_embeddings` tokenizes a chunk, runs it through the model with gradient tracking disabled (we're not training anything, so this saves memory), and mean-pools `last_hidden_state` across every token to collapse the sequence into one fixed-size vector — the exact pooling operation discussed in "Why We Need Chunking" above, and the reason chunk size directly affects embedding quality.

Applying this to every chunk in `splits` produces `text_embedded`: a list of vectors aligned 1:1 with `splits`, ready to be written into a vector database.

In [18]:
def get_text_embeddings(text):
    inputs = text_tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = text_model(**inputs)
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings[0].detach().numpy()
 
text_embedded = [get_text_embeddings(document.page_content) for document in splits]

## Setting Up a Vector Database

With embeddings in hand, we need somewhere to store them for fast similarity search — this is Qdrant's job. `QdrantClient(":memory:")` runs Qdrant in-process for this notebook: no server to stand up, but also no persistence — the index disappears the moment the kernel restarts. Swap `":memory:"` for a file path or a running Qdrant server URL once this moves past prototyping.

In [19]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")

## Indexing Chunks into Qdrant

`index_documents` first creates a collection sized to match the embedding dimensionality, configured for cosine distance — the standard choice for text embeddings, since it measures the angle between vectors (their direction/meaning) rather than magnitude.

It then uploads every chunk as a "point": a random UUID as its id, its embedding vector, and a payload carrying the original text and metadata. That payload is what we get back from a search — it's how retrieval can hand the generation step actual text and citations, not just a bare vector.

In [20]:
from uuid import uuid4
import numpy as np
from qdrant_client import models

def index_documents(client, collection_name, documents, embeddings, distance=models.Distance.COSINE):
    vector_size = len(embeddings[0])

    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(size=vector_size, distance=distance),
        )

    client.upload_points(
        collection_name=collection_name,
        points=[
            models.PointStruct(
                id=str(uuid4()),
                vector=np.array(embeddings[idx]),
                payload={
                    "metadata": doc.metadata,
                    "content": doc.page_content,
                },
            )
            for idx, doc in enumerate(documents)
        ],
    )

index_documents(client, "research_collection", splits, text_embedded)

## Retrieving Relevant Chunks

This is the retrieval step of RAG. `query_qdrant` embeds the user's query with the *same* embedding model used for the documents — this is non-negotiable: query and document vectors must come from the same embedding space to be comparable at all — then asks Qdrant for the `limit` nearest points by cosine similarity.

Each hit's payload (content + metadata) is unpacked into a plain result dict, ready to be handed to the generation step as context.

In [21]:
def query_qdrant(query, qdrant_client, limit=5):
    query_em = get_text_embeddings(query)
    
    text_hits = qdrant_client.query_points(
        collection_name="research_collection",
        query=query_em,
        limit=limit
    ).points
    
    results = []
    for i, point in enumerate(text_hits):
        results.append({
            'source_id': i + 1,
            'content': point.payload['content'],
            'metadata': point.payload['metadata']
        })
    
    return results

## Generating the Final Answer

This is the generation step. Retrieved chunks become the "Context" folded into a prompt that instructs the model to answer using citations back to the source papers — the same architecture sketched in the intro at the top of this notebook.

`generate_answer` reuses the Claude client already configured in `llm_model.py` at the repo root, rather than standing up a separate model, and streams tokens as they're generated so a reader watching the notebook run sees the answer appear incrementally instead of waiting on the full response.

In [22]:
import sys
sys.path.append("..")

from llm_model import llm

def generate_answer(query):
    sources = query_qdrant(query, client)

    prompt = f"""
    Based on the following query, generate a comprehensive answer.
    Include citations [1][2] and mention authors, paper titles, and venues.
    Explain concepts clearly.
 
    Query: "{query}"
    Context: "{sources}"
    
    Return in Markdown format.
    """
    
    response = ""
    for chunk in llm.stream(prompt):
        print(chunk.content, end="", flush=True)
        response += chunk.content

    return response, sources

## Running the Pipeline End to End

With chunking, embedding, indexing, retrieval, and generation all wired up, we can run a real question through the whole thing — from raw text to a cited, Markdown-formatted answer grounded in the papers we indexed.

In [23]:
query = "an autoassociative neural network with dynamic synapses"
 
response, sources=generate_answer(query)

# Autoassociative Neural Networks with Dynamic Synapses

## Overview

An autoassociative neural network with dynamic synapses represents an advanced neural network architecture that incorporates activity-dependent synaptic mechanisms. This approach significantly enhances the network's ability to retrieve and switch between stored patterns of information [1].

## Key Characteristics

### Network Architecture and Dynamics

According to Torres, Cortés, Marro, and Kappen in their 2007 paper "Attractor neural networks with activity-dependent synapses: The role of synaptic facilitation" published in *Neurocomputing*, autoassociative networks with dynamic synapses are characterized by:

- **Synaptic Facilitation Mechanism**: The network incorporates activity-dependent synapses that include a facilitating mechanism, allowing synapses to strengthen over repeated activation [1]
- **Mean-Field Framework**: A general mean-field theoretical approach can be applied to study how different parameters 

## Evaluating the RAG Pipeline

A good final answer can hide a broken middle step — the LLM is skilled enough to sound confident even when it was handed the wrong context, or no context at all. To actually trust this pipeline, each stage needs to be checked independently:

- **Chunking** has no direct metric of its own. It's evaluated indirectly, through its effect on retrieval — sweep `chunk_size` and watch whether retrieval quality improves or degrades.
- **Retrieval** needs a labeled set of "this question is answered by that chunk" pairs, so we can check whether the right chunk actually comes back in the top-k results.
- **Generation** needs to be checked against the context it was actually given, not against vibes — did it stick to what the retrieved chunks support, and did it address the question that was asked?

We don't have human-labeled question/chunk pairs for this dataset, so the next section builds a synthetic evaluation set with the LLM itself, then uses it to score retrieval, chunking, and generation in turn.

## Building a Synthetic Evaluation Set

For a sample of chunks, we ask the LLM to write one specific question that chunk answers. Each `(question, source_chunk)` pair becomes a ground-truth label: retrieval "succeeds" on that question if the source chunk's paper shows up in the top-k results, and generation is later checked against the same source.

We key ground truth on `paper_id` (from each chunk's metadata) rather than the exact chunk text. That matters once we start changing `chunk_size` in the next section — the chunk boundaries shift, so the original chunk's exact text may no longer exist anywhere in a re-chunked index, but the paper it came from still does.

This isn't as rigorous as human-labeled questions, but it's enough to catch a badly tuned chunk size or a broken embedding/retrieval step.

In [24]:
import random

def build_synthetic_eval_set(documents, n=20, seed=42):
    random.seed(seed)
    sample = random.sample(documents, min(n, len(documents)))

    eval_set = []
    for doc in sample:
        prompt = f"""
        Write exactly one specific question that is fully answered by the passage below.
        Return only the question, nothing else.

        Passage: "{doc.page_content}"
        """
        question = llm.invoke(prompt).content.strip()
        eval_set.append({
            "question": question,
            "source_content": doc.page_content,
            "paper_id": doc.metadata["paper_id"],
        })

    return eval_set

eval_set = build_synthetic_eval_set(splits, n=20)
print(f"Built {len(eval_set)} synthetic eval questions")

Built 20 synthetic eval questions


## Evaluating Retrieval

For each synthetic question, run it through `query_qdrant` and check whether any of the top-k results traces back to the same `paper_id` the question was generated from.

- **Recall@k** — the fraction of questions where the correct paper appears anywhere in the top-k results. Since the source chunk is guaranteed to already be indexed, a low score here means the embedding model or similarity search is failing to place semantically matching text near each other in vector space — not that the information doesn't exist.
- **MRR (Mean Reciprocal Rank)** — rewards ranking a correct hit near the *top* of the results, not just anywhere in the top-k. A system with high recall but low MRR is finding the right chunk, but burying it under less-relevant ones.

In [25]:
def evaluate_retrieval(eval_set, qdrant_client, k=5):
    hits, reciprocal_ranks = 0, []

    for item in eval_set:
        results = query_qdrant(item["question"], qdrant_client, limit=k)
        rank = next(
            (i + 1 for i, r in enumerate(results) if r["metadata"]["paper_id"] == item["paper_id"]),
            None,
        )
        if rank is not None:
            hits += 1
            reciprocal_ranks.append(1 / rank)
        else:
            reciprocal_ranks.append(0)

    return {
        "recall@k": hits / len(eval_set),
        "mrr": sum(reciprocal_ranks) / len(eval_set),
    }

retrieval_scores = evaluate_retrieval(eval_set, client, k=5)
print(retrieval_scores)

{'recall@k': 1.0, 'mrr': 1.0}


## Evaluating Generation with an LLM Judge

Generation is graded on two axes that don't require a ground-truth answer to write:

- **Faithfulness** — does every claim in the generated answer actually trace back to the retrieved context, or is the model adding facts it wasn't given?
- **Relevancy** — does the answer actually address the question that was asked, rather than just reciting the context?

The standard tool for this is `ragas`, but its LLM wrapper still imports `ChatVertexAI` from a `langchain_community` module that no longer exists in the `langchain-community` version this repo runs (it requires `langchain-community<0.4`, which in turn requires `langchain<1.0.0` — a downgrade that would break every other notebook in this repo built on LangChain's v1 API). Rather than downgrade `langchain` repo-wide for one eval cell, we implement the same idea directly: ask the LLM itself (the same `llm` from `llm_model.py` used for generation) to score each `(question, context, answer)` triple against a fixed rubric, and average the scores. This is exactly what `ragas`'s metrics do under the hood — an LLM-as-judge — just without the extra dependency.

In [26]:
import json
import re

def parse_llm_json(message_content):
    match = re.search(r"[\{\[].*[\}\]]", message_content, re.DOTALL)
    json_str = match.group(0) if match else message_content
    return json.loads(json_str)

def judge_answer(question, context, answer):
    judge_prompt = f"""
    You are evaluating a generated answer against the context it was given.

    Question: "{question}"
    Context: "{context}"
    Answer: "{answer}"

    Score the answer on two dimensions, each from 1 (worst) to 5 (best):
    1. faithfulness: does every claim in the answer come from the context, with no invented facts?
    2. relevancy: does the answer directly address the question?

    Respond with only a JSON object in this exact format, nothing else:
    {{"faithfulness": <int>, "relevancy": <int>}}
    """
    judgment = llm.invoke(judge_prompt).content
    return parse_llm_json(judgment)

def evaluate_generation(eval_set, n=10):
    scores = []
    for item in eval_set[:n]:
        answer, sources = generate_answer(item["question"])
        context = "\n\n".join(s["content"] for s in sources)
        scores.append(judge_answer(item["question"], context, answer))

    return {
        "faithfulness": sum(s["faithfulness"] for s in scores) / len(scores),
        "relevancy": sum(s["relevancy"] for s in scores) / len(scores),
    }

generation_scores = evaluate_generation(eval_set, n=10)
print(generation_scores)

# Computational Complexity of Coupled Task Scheduling with Unit Processing Times

## Problem Definition

The coupled task scheduling problem with unit processing times, fixed gap length, and strict precedence constraints involves scheduling tasks on a single processor where:

- **Processing times**: All tasks have equal length of 1 unit
- **Gap constraint**: There is an exact mandatory gap of fixed length *h* between the processing of each task
- **Precedence constraints**: Tasks must be scheduled according to strict precedence relationships
- **Objective**: Minimize the total schedule length (makespan)

## Complexity Classification: NP-Hard

According to Blazewicz et al. (2001) in their paper "A note on the complexity of scheduling coupled tasks on a single processor" published in the *Journal of the Brazilian Computer Society* [1], **this problem is NP-hard**.

### Key Finding

The authors demonstrate that the general coupled task scheduling problem on a single processor with unit pr

## Building a Held-Out Test Set

The evaluation above has a flaw worth naming: `eval_set` was built by asking the LLM to write questions about the *same 100 documents* (`data[:100]`) that were indexed. Even keying on `paper_id` instead of exact chunk text doesn't fix this — the content those questions are about was fully present in the index the whole time. A retrieval system could look far better here than it actually is.

A proper held-out test uses content the pipeline has never seen: index a second, previously-unused batch of 100 documents (`data[100:200]`) into the *same* collection, then generate the evaluation questions exclusively from that new batch. Retrieval now has to find the right chunk among a larger, more realistic index, and the questions are about documents that played no part in building or tuning anything above.

In [27]:
held_out_documents = data[100:200]

held_out_splits = recursive_split_documents(held_out_documents)
held_out_embeddings = [get_text_embeddings(doc.page_content) for doc in held_out_splits]

index_documents(client, "research_collection", held_out_splits, held_out_embeddings)
print(f"Indexed {len(held_out_splits)} additional held-out chunk(s) from {len(held_out_documents)} document(s)")

Indexed 144 additional held-out chunk(s) from 100 document(s)


## Running the Held-Out Evaluation

With the held-out batch indexed, build questions from it exclusively and rerun the same retrieval and generation metrics used above. The index now also contains the original 100 documents as distractors, so a correct retrieval has to outrank them, not just outrank an empty or trivial search space — the closest thing to a real "unseen queries against a live index" test this notebook can do without external human-labeled data.

In [28]:
held_out_eval_set = build_synthetic_eval_set(held_out_splits, n=20)
print(f"Built {len(held_out_eval_set)} held-out eval question(s)")

held_out_retrieval_scores = evaluate_retrieval(held_out_eval_set, client, k=5)
print("Held-out retrieval:", held_out_retrieval_scores)

held_out_generation_scores = evaluate_generation(held_out_eval_set, n=10)
print("Held-out generation:", held_out_generation_scores)

Built 20 held-out eval question(s)
Held-out retrieval: {'recall@k': 0.9, 'mrr': 0.9}
# Key Components of the Hera Methodology for Designing Web Information Systems

## Overview

Hera is a model-driven design methodology specifically created to support the design of Web Information Systems (WIS). According to Houben et al. (2003), WIS utilize web paradigm and technologies to retrieve information from sources connected to the Web and present this information in a web or hypermedia format to users [1].

## Core Components

The Hera methodology distinguishes three fundamental components in its design approach [1]:

### 1. **Integration Model**
The integration model covers the different aspects of integrating data from multiple web sources. This component addresses how heterogeneous information sources are brought together within the system architecture.

### 2. **Data Gathering**
This component focuses on the collection and retrieval of information from various sources connected to the Web

In [29]:
from uuid import uuid4
import numpy as np
from qdrant_client import models

def index_documents(client, collection_name, documents, embeddings, distance=models.Distance.COSINE):
    vector_size = len(embeddings[0])

    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config=models.VectorParams(size=vector_size, distance=distance),
        )

    client.upload_points(
        collection_name=collection_name,
        points=[
            models.PointStruct(
                id=str(uuid4()),
                vector=np.array(embeddings[idx]),
                payload={
                    "metadata": doc.metadata,
                    "content": doc.page_content,
                },
            )
            for idx, doc in enumerate(documents)
        ],
    )
print("index_documents redefined (idempotent)")

index_documents redefined (idempotent)
